# Dự đoán Rủi ro Tín dụng (Credit Risk Prediction)

## 1. Introduction

- **Bài toán:** Phân loại các khoản vay thành hai nhóm **rủi ro thấp (`low_risk`)** và **rủi ro cao (`high_risk`)** dựa trên thông tin hồ sơ vay và lịch sử tín dụng.
- **Động lực:** Trong thực tế tài chính, số lượng khoản vay vỡ nợ/trễ hạn chiếm tỷ lệ rất nhỏ (~0.5%) nhưng gây thiệt hại lớn nếu bỏ sót. Mô hình phân loại thông thường dễ bị thiên lệch về lớp đa số (`low_risk`).
- **Mục tiêu:** Xây dựng quy trình Machine Learning hoàn chỉnh cho dữ liệu mất cân bằng cực đoan bằng các thuật toán Ensemble chuyên biệt (`BalancedRandomForestClassifier` và `EasyEnsembleClassifier`), tối ưu khả năng phát hiện khoản vay rủi ro cao (`Recall` trên `high_risk` và `Balanced Accuracy`).

### Khởi tạo môi trường và thư viện
Nạp các thư viện xử lý dữ liệu (`numpy`, `pandas`), trực quan hóa (`matplotlib`, `seaborn`), đánh giá mô hình (`scikit-learn`) và học máy cho dữ liệu mất cân bằng (`imbalanced-learn`).

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import confusion_matrix
from imblearn.metrics import classification_report_imbalanced

## 2. Dataset (20%) — Khai báo đặc trưng và biến mục tiêu

- **Nguồn dữ liệu:** Bộ dữ liệu khoản vay thực tế của **LendingClub (Quý 1/2019 — `LoanStats_2019Q1.csv`)**.
- **Lựa chọn đặc trưng:** Từ 144 cột gốc, chọn lọc **86 cột** liên quan trực tiếp đến đặc điểm khoản vay (`loan_amnt`, `int_rate`, `installment`), năng lực tài chính (`annual_inc`, `dti`, `home_ownership`) và lịch sử tín dụng (`delinq_2yrs`, `revol_bal`, `total_rec_prncp`,...).
- **Biến mục tiêu (`target`):** `loan_status` (trạng thái khoản vay).

In [ ]:
columns = [
    "loan_amnt", "int_rate", "installment", "home_ownership",
    "annual_inc", "verification_status", "issue_d", "loan_status",
    "pymnt_plan", "dti", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_bal", "total_acc",
    "initial_list_status", "out_prncp", "out_prncp_inv", "total_pymnt",
    "total_pymnt_inv", "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee", "last_pymnt_amnt", "next_pymnt_d",
    "collections_12_mths_ex_med", "policy_code", "application_type", "acc_now_delinq",
    "tot_coll_amt", "tot_cur_bal", "open_acc_6m", "open_act_il",
    "open_il_12m", "open_il_24m", "mths_since_rcnt_il", "total_bal_il",
    "il_util", "open_rv_12m", "open_rv_24m", "max_bal_bc",
    "all_util", "total_rev_hi_lim", "inq_fi", "total_cu_tl",
    "inq_last_12m", "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy",
    "bc_util", "chargeoff_within_12_mths", "delinq_amnt", "mo_sin_old_il_acct",
    "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mort_acc",
    "mths_since_recent_bc", "mths_since_recent_inq", "num_accts_ever_120_pd", "num_actv_bc_tl",
    "num_actv_rev_tl", "num_bc_sats", "num_bc_tl", "num_il_tl",
    "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0",
    "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd", "num_tl_90g_dpd_24m",
    "num_tl_op_past_12m", "pct_tl_nvr_dlq", "percent_bc_gt_75", "pub_rec_bankruptcies",
    "tax_liens", "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit",
    "total_il_high_credit_limit", "hardship_flag", "debt_settlement_flag"
]

target = ["loan_status"]

### Tiền xử lý và làm sạch dữ liệu (Data Cleaning & Labeling)

Cell này thực hiện chuỗi bước làm sạch dữ liệu:
1. Đọc file CSV (bỏ dòng ghi chú đầu tiên `skiprows=1` và 2 dòng tổng kết cuối file `[:-2]`).
2. Lọc đúng 86 cột đã định nghĩa, loại bỏ các cột rỗng hoàn toàn và các dòng có giá trị thiếu (`NaN`).
3. Loại bỏ các khoản vay có trạng thái `Issued` (mới phát hành, chưa có lịch sử trả nợ).
4. Chuyển cột lãi suất `int_rate` từ chuỗi có ký tự `%` sang số thực (ví dụ: `17.19%` → `0.1719`).
5. Gán nhãn nhị phân cho biến mục tiêu `loan_status`:
   - `Current` → **`low_risk`**
   - `Late (31-120 days)`, `Late (16-30 days)`, `Default`, `In Grace Period` → **`high_risk`**

In [12]:
# Load the data
file_path = Path(r'D:\credit-risk\credit-risk\LoanStats_2019Q1.csv')
df = pd.read_csv(file_path, skiprows=1, low_memory=False)[:-2]
df = df.loc[:, columns].copy()

# Drop the null columns where all values are null
df = df.dropna(axis='columns', how='all')

# Drop the null rows
df = df.dropna()

# Remove the `Issued` loan status
issued_mask = df['loan_status'] != 'Issued'
df = df.loc[issued_mask]

# convert interest rate to numerical
df['int_rate'] = df['int_rate'].str.replace('%', '')
df['int_rate'] = df['int_rate'].astype('float') / 100


# Convert the target column values to low_risk and high_risk based on their values
x = {'Current': 'low_risk'}   
df = df.replace(x)

x = dict.fromkeys(['Late (31-120 days)', 'Late (16-30 days)', 'Default', 'In Grace Period'], 'high_risk')    
df = df.replace(x)

df.reset_index(inplace=True, drop=True)

df.head()

C:\Users\User\AppData\Local\Temp\ipykernel_22488\928680940.py:28: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.reset_index(inplace=True, drop=True)


,loan_amnt,int_rate,installment,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,dti,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,debt_settlement_flag
0,10500.0,0.1719,375.35,RENT,66000.0,Source Verified,Mar-2019,low_risk,n,27.24,...,85.7,100.0,0.0,0.0,65687.0,38199.0,2000.0,61987.0,N,N
1,25000.0,0.2000,929.09,MORTGAGE,105000.0,Verified,Mar-2019,low_risk,n,20.23,...,91.2,50.0,1.0,0.0,271427.0,60641.0,41200.0,49197.0,N,N
2,20000.0,0.2000,529.88,MORTGAGE,56000.0,Verified,Mar-2019,low_risk,n,24.26,...,66.7,50.0,0.0,0.0,60644.0,45684.0,7500.0,43144.0,N,N
3,10000.0,0.1640,353.55,RENT,92000.0,Verified,Mar-2019,low_risk,n,31.44,...,100.0,50.0,1.0,0.0,99506.0,68784.0,19700.0,76506.0,N,N
4,22000.0,0.1474,520.39,MORTGAGE,52000.0,Not Verified,Mar-2019,low_risk,n,18.76,...,100.0,0.0,0.0,0.0,219750.0,25919.0,27600.0,20000.0,N,N


### Mã hóa đặc trưng & Chia tập dữ liệu (Feature Encoding & Stratified Split)

- **One-Hot Encoding (`pd.get_dummies`):** Chuyển đổi 9 biến phân loại (`home_ownership`, `verification_status`, `issue_d`, `pymnt_plan`, `initial_list_status`, `next_pymnt_d`, `debt_settlement_flag`, `application_type`, `hardship_flag`) sang dạng số nhị phân với `drop_first=True` để tránh đa cộng tuyến.
- **Chia tập Train / Test (75% / 25%):** Sử dụng `train_test_split` với `stratify=y` nhằm bảo toàn tỷ lệ phân bố lớp mất cân bằng (~99.49% `low_risk` vs ~0.51% `high_risk`) giữa tập huấn luyện và tập kiểm thử.

In [13]:
# Create our features
X_df = df.drop(columns=['loan_status'])
X = pd.get_dummies(X_df, columns=["home_ownership","verification_status","issue_d",
                                  "pymnt_plan","initial_list_status","next_pymnt_d","debt_settlement_flag",
                                  "application_type","hardship_flag"], drop_first = True)

# Create our target
y = df['loan_status'].to_frame()
# Check the balance of our target values
y['loan_status'].value_counts()
# Split the X and y into X_train, X_test, y_train, y_test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, stratify=y)

## 3. Methodology  — Chuẩn hóa dữ liệu & Mô hình 1: Balanced Random Forest Classifier

### Chuẩn hóa đặc trưng (`StandardScaler`)
Đưa các đặc trưng số về cùng thang đo (trung bình $\mu = 0$, độ lệch chuẩn $\sigma = 1$). Bộ chuẩn hóa chỉ được `fit` trên tập `X_train` rồi `transform` sang `X_test` để tránh rò rỉ dữ liệu (*data leakage*).

### Giới thiệu mô hình: `BalancedRandomForestClassifier`
- **Nguyên lý hoạt động:** Là biến thể của Random Forest dành riêng cho dữ liệu mất cân bằng. Trước khi huấn luyện mỗi cây quyết định (*Decision Tree*), thuật toán thực hiện **Random Under-Sampling** trên lớp đa số (`low_risk`) sao cho mẫu bootstrap có số lượng cân bằng với lớp thiểu số (`high_risk`).
- **Ưu điểm:**
  - Khắc phục tình trạng thiên lệch lớp mà không làm mất quá nhiều thông tin (vì mỗi cây nhìn thấy một tập con ngẫu nhiên khác nhau của lớp đa số).
  - Cung cấp chỉ số **Feature Importance**, giúp xác định các yếu tố ảnh hưởng mạnh nhất đến rủi ro tín dụng.
- **Cấu hình thí nghiệm:** `n_estimators=1000` (1000 cây quyết định), `random_state=1` để đảm bảo khả năng tái lập kết quả.

In [14]:
# Scale data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaler = scaler.fit(X_train)
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

# Resample the training data with the RandomOversampler
from imblearn.ensemble import BalancedRandomForestClassifier

brfc = BalancedRandomForestClassifier(n_estimators =1000, random_state=1)
model = brfc.fit(X_train_scaled, y_train)
BalancedRandomForestClassifier()
# Calculate the balanced accuracy score
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

predictions = model.predict(X_test_scaled)
accuracy_score(y_test, predictions)
# List the features sorted in descending order by feature importance
sorted(zip(model.feature_importances_, X.columns), reverse=True)[:20]

d:\credit-risk\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[(np.float64(0.07534372086138796), 'total_rec_prncp'),
 (np.float64(0.06462151993297456), 'total_rec_int'),
 (np.float64(0.06449157587594419), 'last_pymnt_amnt'),
 (np.float64(0.06048304491088711), 'total_pymnt_inv'),
 (np.float64(0.05835183087441404), 'total_pymnt'),
 (np.float64(0.028724328635404656), 'int_rate'),
 (np.float64(0.023908819163303528), 'issue_d_Jan-2019'),
 (np.float64(0.017687888940215665), 'dti'),
 (np.float64(0.017554675877196328), 'installment'),
 (np.float64(0.016268370374273755), 'annual_inc'),
 (np.float64(0.016124083036875847), 'out_prncp_inv'),
 (np.float64(0.015982936281149072), 'issue_d_Mar-2019'),
 (np.float64(0.015880726192516566), 'out_prncp'),
 (np.float64(0.01572088794013338), 'max_bal_bc'),
 (np.float64(0.015535576326480382), 'mths_since_recent_inq'),
 (np.float64(0.01502440490329204), 'revol_bal'),
 (np.float64(0.014729355895706337), 'il_util'),
 (np.float64(0.014486326229614854), 'bc_util'),
 (np.float64(0.014317309781454323), 'tot_hi_cred_lim'),
 (np

### Mô hình 2: Easy Ensemble Classifier (`EasyEnsembleClassifier`)

- **Nguyên lý hoạt động:** Kết hợp kỹ thuật **Bagging** và **Boosting (AdaBoost)**. Thuật toán tạo ra `n_estimators` tập con cân bằng bằng cách lấy ngẫu nhiên dưới mẫu (*Random Under-Sampling*) từ lớp đa số kết hợp với toàn bộ lớp thiểu số, sau đó huấn luyện một bộ phân loại **AdaBoost** độc lập trên từng tập con và tổng hợp dự đoán cuối cùng.
- **Ưu điểm:**
  - Tận dụng sức mạnh giảm sai số (*bias*) của AdaBoost đồng thời giảm phương sai (*variance*) nhờ Bagging trên nhiều tập con.
  - Thường đạt **Recall** và **Balanced Accuracy** rất cao trên bài toán phát hiện rủi ro/gian lận có tỷ lệ mất cân bằng cực đoan.
- **Cấu hình thí nghiệm:** `estimator=None` (mặc định `AdaBoostClassifier`), `n_estimators=100`, `sampling_strategy='auto'`, `replacement=False`, `random_state=1`.

In [21]:
# Train the EasyEnsembleClassifier
# Train the Classifier
from imblearn.ensemble import EasyEnsembleClassifier

model = EasyEnsembleClassifier(
    estimator=None,
    n_estimators=100,
    n_jobs=1,
    random_state=1,
    replacement=False,
    sampling_strategy='auto',
    verbose=0,
    warm_start=False,
)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

EasyEnsembleClassifier()
# Calculated the balanced accuracy score
from sklearn.metrics import balanced_accuracy_score
predictions = model.predict(X_test_scaled)
balanced_accuracy_score(y_test, predictions)

d:\credit-risk\.venv\Lib\site-packages\sklearn\ensemble\_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.8999268767298789

## 4. Evaluation  — Đánh giá mô hình trên dữ liệu mất cân bằng

Với dữ liệu có **99.5% `low_risk`** và chỉ **0.5% `high_risk`**, chỉ số `Accuracy` thông thường dễ gây ngộ nhận (một mô hình đoán tất cả là `low_risk` vẫn đạt 99.5% accuracy nhưng vô dụng). Do đó, ta đánh giá bằng:
- **Balanced Accuracy:** Trung bình cộng của Recall trên cả 2 lớp.
- **Confusion Matrix:** Quan sát trực tiếp số lượng dự đoán đúng/sai của từng lớp (đặc biệt là *False Negative* — bỏ sót khoản vay rủi ro cao).
- **Classification Report (Precision, Recall, F1-score):** Trong quản trị rủi ro tín dụng, **Recall trên lớp `high_risk`** là chỉ số then chốt vì chi phí bỏ sót một khoản vay vỡ nợ cao hơn nhiều so với chi phí kiểm tra thêm một hồ sơ bị cảnh báo nhầm.

In [22]:
# Display the confusion matrix
confusion_matrix(y_test, predictions)
# Print the imbalanced classification report
from imblearn.metrics import classification_report_imbalanced
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

   high_risk       0.07      0.86      0.12        87
    low_risk       1.00      0.94      0.97     17118

    accuracy                           0.94     17205
   macro avg       0.53      0.90      0.54     17205
weighted avg       0.99      0.94      0.96     17205



## 5. Conclusion & Future Work

### Kết luận
- Mô hình **EasyEnsembleClassifier** đạt **Balanced Accuracy = 89.99%** và **Overall Accuracy = 94%** trên tập kiểm thử (17,205 mẫu).
- Trên lớp mục tiêu **`high_risk`** (87 mẫu), mô hình đạt **Recall = 0.86** (phát hiện được khoảng 86% các khoản vay có nguy cơ rủi ro cao) và **Recall = 0.94** trên lớp **`low_risk`** (Precision = 1.00).
- **Hạn chế cần lưu ý:** Precision trên lớp `high_risk` còn thấp (**0.07**), phản ánh sự đánh đổi (*Precision–Recall trade-off*) điển hình khi mất cân bằng lớp ở mức ~197:1.
- Các yếu tố quan trọng nhất từ `BalancedRandomForestClassifier` gồm: `total_rec_prncp` (7.53%), `total_rec_int` (6.46%), `last_pymnt_amnt` (6.45%), `total_pymnt_inv` (6.05%), `total_pymnt` (5.84%) và `int_rate` (2.87%).

### Hướng phát triển tiếp theo (Future Work)
1. **Kiểm soát rò rỉ thông tin theo thời gian:** Loại bỏ các biến phát sinh sau khi giải ngân (`total_rec_prncp`, `last_pymnt_amnt`,...) nếu xây dựng mô hình chấm điểm tín dụng tại thời điểm xét duyệt hồ sơ (*origination scoring*).
2. **Thử nghiệm thêm mô hình & tối ưu ngưỡng:** So sánh với XGBoost / LightGBM kết hợp `scale_pos_weight`, tối ưu ngưỡng xác suất (*threshold tuning*) để cải thiện Precision/F1 mà vẫn giữ Recall cao.
3. **Giải thích mô hình & triển khai:** Tích hợp **SHAP** để giải thích quyết định cho từng hồ sơ vay và đóng gói thành API/Web App hỗ trợ thẩm định tín dụng.
